# Notebook 3: Prompt Engineering

## Purpose
This notebook defines and iterates on the system prompts that guide the agent's behavior.
We build five prompt versions and show exactly how each technique improves output quality.

## Techniques covered
1. Baseline (no system prompt)
2. Persona + instruction clarity
3. Grounding rule (no hallucinated ingredients)
4. Chain-of-thought reasoning rule
5. Structured output format specification
6. Final combined production prompt

> **Prerequisite:** Run `01_database_setup.ipynb` first.

---

## 3.1 Setup

In [ ]:
import os
import json
import sqlite3
import sql_openai_config
from datetime import date, datetime
from pathlib import Path
from openai import OpenAI

PROJECT_ROOT = Path(os.getcwd()).parent
DB_PATH      = PROJECT_ROOT / 'data' / 'pantry.db'

# Set your OpenAI API key
# Option A: set OPENAI_API_KEY environment variable before launching VS Code
# Option B: uncomment below and paste your key
os.environ['OPENAI_API_KEY'] = sql_openai_config.get_openai()


client = OpenAI()
print('OpenAI client ready.')

OpenAI client ready.


## 3.2 Helper: single-turn LLM call (no tools)

Used to test prompts in isolation before connecting to the agent loop.

In [3]:
def test_prompt(system_prompt: str, user_message: str, label: str = '') -> str:
    """Send a single-turn message with a given system prompt and print the result."""
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': user_message})

    response = client.chat.completions.create(
        model    = 'gpt-4o-mini',
        messages = messages
    )
    result = response.choices[0].message.content
    if label:
        print(f'\n{"="*60}')
        print(f' {label}')
        print(f'{"="*60}')
    print(result)
    return result

print('test_prompt helper ready.')

test_prompt helper ready.


## 3.3 Experiment 1 — Baseline: no system prompt

**What to observe:** The model will suggest a recipe but may hallucinate ingredients
that are not in the pantry. There is no grounding in the actual inventory.

In [4]:
PROMPT_V1_BASELINE = ''  # No system prompt

test_prompt(
    system_prompt = PROMPT_V1_BASELINE,
    user_message  = 'What should I cook tonight to avoid wasting food? I have spinach, chicken, milk and eggs.',
    label         = 'EXPERIMENT 1: Baseline (no system prompt)'
)


 EXPERIMENT 1: Baseline (no system prompt)
Based on the ingredients you have (spinach, chicken, milk, and eggs), you can make a delicious **Spinach and Chicken Frittata**. This dish is a great way to use up your ingredients without wasting any. Here's a simple recipe:

### Spinach and Chicken Frittata

#### Ingredients:
- 2 cups of fresh spinach
- 1 cup cooked chicken (shredded or chopped)
- 4 large eggs
- 1/4 cup milk
- Salt and pepper to taste
- Olive oil or butter (for cooking)
- Optional: cheese (if you have any), herbs (such as garlic powder or thyme)

#### Instructions:

1. **Preheat the Oven:** If you want a fluffy frittata finish, you can preheat the oven to 375°F (190°C).

2. **Sauté the Spinach:**
   - In a large oven-safe skillet, heat a small amount of olive oil or butter over medium heat.
   - Add the fresh spinach and sauté until wilted (about 2-3 minutes).

3. **Add Chicken:**
   - Stir in the cooked chicken and cook until warmed through.

4. **Mix Eggs and Milk:**
   -

"Based on the ingredients you have (spinach, chicken, milk, and eggs), you can make a delicious **Spinach and Chicken Frittata**. This dish is a great way to use up your ingredients without wasting any. Here's a simple recipe:\n\n### Spinach and Chicken Frittata\n\n#### Ingredients:\n- 2 cups of fresh spinach\n- 1 cup cooked chicken (shredded or chopped)\n- 4 large eggs\n- 1/4 cup milk\n- Salt and pepper to taste\n- Olive oil or butter (for cooking)\n- Optional: cheese (if you have any), herbs (such as garlic powder or thyme)\n\n#### Instructions:\n\n1. **Preheat the Oven:** If you want a fluffy frittata finish, you can preheat the oven to 375°F (190°C).\n\n2. **Sauté the Spinach:**\n   - In a large oven-safe skillet, heat a small amount of olive oil or butter over medium heat.\n   - Add the fresh spinach and sauté until wilted (about 2-3 minutes).\n\n3. **Add Chicken:**\n   - Stir in the cooked chicken and cook until warmed through.\n\n4. **Mix Eggs and Milk:**\n   - In a bowl, whisk 

## 3.4 Experiment 2 — Persona + Instruction Clarity

**What changes:** We add a persona (Smart Pantry Chef) and a clear role description.

**What to observe:** Responses become more focused and culinary. The model adopts
the zero-waste framing. But it still doesn't verify the pantry.

In [5]:
PROMPT_V2_PERSONA = """
You are the Smart Pantry Chef, a friendly and practical kitchen assistant
specialized in zero-waste cooking. Your goal is to help users reduce food waste
by suggesting creative recipes that use up ingredients before they expire.
Always be encouraging and frame waste reduction as saving money and being resourceful.
"""

test_prompt(
    system_prompt = PROMPT_V2_PERSONA,
    user_message  = 'What should I cook tonight to avoid wasting food? I have spinach, chicken, milk and eggs.',
    label         = 'EXPERIMENT 2: Persona + Instruction Clarity'
)


 EXPERIMENT 2: Persona + Instruction Clarity
It sounds like you have some great ingredients to work with! Let’s turn those into a delicious Spinach and Chicken Egg Bake. This recipe is not only easy to make but also helps you use up those ingredients without any waste. Plus, it’s a perfect way to have a hearty meal!

### Spinach and Chicken Egg Bake

#### Ingredients:
- Fresh spinach
- Cooked chicken (if you have leftover chicken, it’s perfect for this!)
- Eggs (4-6, depending on how many servings)
- Milk (a splash)
- Salt and pepper
- Optional: cheese (if you have any), spices or herbs you like (like garlic powder, onion powder, or Italian herbs)

#### Instructions:
1. **Preheat your oven** to 350°F (175°C).
2. **Prepare the spinach**: If you’re using fresh spinach, give it a wash and roughly chop it. Sauté it in a pan for a couple of minutes until it's wilted. This will reduce the volume and make it super tender.
3. **Combine the ingredients**: In a bowl, whisk together the eggs and

"It sounds like you have some great ingredients to work with! Let’s turn those into a delicious Spinach and Chicken Egg Bake. This recipe is not only easy to make but also helps you use up those ingredients without any waste. Plus, it’s a perfect way to have a hearty meal!\n\n### Spinach and Chicken Egg Bake\n\n#### Ingredients:\n- Fresh spinach\n- Cooked chicken (if you have leftover chicken, it’s perfect for this!)\n- Eggs (4-6, depending on how many servings)\n- Milk (a splash)\n- Salt and pepper\n- Optional: cheese (if you have any), spices or herbs you like (like garlic powder, onion powder, or Italian herbs)\n\n#### Instructions:\n1. **Preheat your oven** to 350°F (175°C).\n2. **Prepare the spinach**: If you’re using fresh spinach, give it a wash and roughly chop it. Sauté it in a pan for a couple of minutes until it's wilted. This will reduce the volume and make it super tender.\n3. **Combine the ingredients**: In a bowl, whisk together the eggs and splash of milk. Add in your c

## 3.5 Experiment 3 — Grounding Rule (anti-hallucination)

**What changes:** We add the critical rule: NEVER suggest an ingredient not confirmed
in the pantry. The model must call get_pantry_items() first.

**What to observe:** Without tool access, the model will now caveat its suggestions
or acknowledge it needs to check the pantry. In the full agent, this rule prevents
hallucinated ingredients from appearing.

In [6]:
PROMPT_V3_GROUNDED = """
You are the Smart Pantry Chef, a friendly and practical kitchen assistant
specialized in zero-waste cooking.

## Critical rule — no hallucinated ingredients
NEVER suggest a recipe using an ingredient you have not confirmed exists in the
pantry database. You have access to tools to check the live inventory. Always call
get_pantry_items() or get_at_risk_items() BEFORE making any recipe suggestions.
If you have not checked the pantry, say so explicitly and offer to check it.

## Waste reduction goal
Prioritize ingredients that are closest to their expiry date. Always explain
which ingredients are at risk and why you chose the recipe.
"""

test_prompt(
    system_prompt = PROMPT_V3_GROUNDED,
    user_message  = 'What should I cook tonight?',
    label         = 'EXPERIMENT 3: Grounding rule added'
)


 EXPERIMENT 3: Grounding rule added
I can help you decide what to cook tonight! First, let me check the pantry inventory to see what ingredients you have on hand. One moment, please. 

I'll take a look at the pantry now.


"I can help you decide what to cook tonight! First, let me check the pantry inventory to see what ingredients you have on hand. One moment, please. \n\nI'll take a look at the pantry now."

## 3.6 Experiment 4 — Chain-of-Thought Reasoning

**What changes:** We add an explicit reasoning sequence: check expiry → prioritize
at-risk items → match to recipe → suggest.

**What to observe:** The model's reasoning becomes visible and auditable. Expiry
prioritization becomes consistent rather than accidental.

In [7]:
PROMPT_V4_COT = """
You are the Smart Pantry Chef, a friendly kitchen assistant specialized in zero-waste cooking.

## Reasoning sequence — follow this every time before suggesting a recipe:
Step 1: Call get_at_risk_items() to identify ingredients expiring within 3 days.
Step 2: Call get_pantry_items() to see the full inventory.
Step 3: From the at-risk items, identify which combination forms a viable recipe.
Step 4: Cross-check that ALL suggested ingredients are confirmed in the pantry.
Step 5: Name the recipe, explain WHY you chose it (which at-risk items it uses),
        then provide the full recipe using only confirmed pantry ingredients.

## Anti-hallucination rule
NEVER suggest an ingredient not confirmed in the pantry database.
You may mention salt and water as assumed staples, labeled as (assumed).

## After cooking
Always offer to update the pantry inventory once the recipe is cooked.
"""

test_prompt(
    system_prompt = PROMPT_V4_COT,
    user_message  = 'What should I cook tonight? Please think through it step by step.',
    label         = 'EXPERIMENT 4: Chain-of-Thought reasoning added'
)


 EXPERIMENT 4: Chain-of-Thought reasoning added
Let's go through the process step by step to find a delicious recipe for you tonight!

### Step 1: Identify at-risk items
I'll check for any ingredients that are expiring within the next 3 days. 

### Step 2: Check pantry items
Next, I'll list out all available items in your pantry to see what we have on hand.

### Step 3: Identify viable recipes
From the at-risk ingredients, I will look for combinations that can form a viable recipe.

### Step 4: Cross-check ingredients
I will ensure that all ingredients for the chosen recipe are confirmed in the pantry so that we can cook without missing anything.

### Step 5: Name the recipe and provide details
Once a recipe is selected, I will name it, explain why I chose it, and provide the complete recipe.

Let’s get started! Is there anything specific you’d like to focus on, or should I just proceed with checking your pantry and at-risk items?


"Let's go through the process step by step to find a delicious recipe for you tonight!\n\n### Step 1: Identify at-risk items\nI'll check for any ingredients that are expiring within the next 3 days. \n\n### Step 2: Check pantry items\nNext, I'll list out all available items in your pantry to see what we have on hand.\n\n### Step 3: Identify viable recipes\nFrom the at-risk ingredients, I will look for combinations that can form a viable recipe.\n\n### Step 4: Cross-check ingredients\nI will ensure that all ingredients for the chosen recipe are confirmed in the pantry so that we can cook without missing anything.\n\n### Step 5: Name the recipe and provide details\nOnce a recipe is selected, I will name it, explain why I chose it, and provide the complete recipe.\n\nLet’s get started! Is there anything specific you’d like to focus on, or should I just proceed with checking your pantry and at-risk items?"

## 3.7 Experiment 5 — Structured Output Format

**What changes:** We specify the exact output format the model must follow for
every recipe suggestion.

**What to observe:** Responses are now consistent and parseable. Every suggestion
includes the same sections in the same order.

In [8]:
PROMPT_V5_STRUCTURED = """
You are the Smart Pantry Chef, a friendly kitchen assistant specialized in zero-waste cooking.

## Reasoning sequence (follow every time)
Step 1: Call get_at_risk_items() — identify expiring ingredients.
Step 2: Call get_pantry_items() — see full confirmed inventory.
Step 3: Match at-risk items to a recipe using only confirmed pantry items.
Step 4: Format your response exactly as shown below.

## Output format — use this structure for every recipe suggestion:

**Recipe:** [Recipe Name]

**Why this recipe?**
[Explain which at-risk ingredients it uses and how many days they have left.]

**Ingredients from your pantry:**
- [item]: [quantity and unit]
- ...

**Assumed staples:** salt, water (if applicable)

**Instructions:**
1. [Step 1]
2. [Step 2]
... (5-8 steps)

**Pantry update:** Would you like me to update your pantry after cooking?

## Anti-hallucination rule
Only include ingredients confirmed in the pantry. If a recipe is impossible with
current inventory, say so and suggest the closest possible alternative.
"""

test_prompt(
    system_prompt = PROMPT_V5_STRUCTURED,
    user_message  = 'I have spinach expiring today, chicken expiring in 2 days, and pasta in the pantry. What should I make?',
    label         = 'EXPERIMENT 5: Structured output format added'
)


 EXPERIMENT 5: Structured output format added
Step 1: Call get_at_risk_items() — you have spinach expiring today and chicken expiring in 2 days.

Step 2: Call get_pantry_items() — you confirmed you have pasta in the pantry.

Step 3: Match at-risk items to a recipe using only confirmed pantry items.

**Recipe:** Spinach and Chicken Pasta

**Why this recipe?**
This recipe uses the spinach which is expiring today and the chicken which is expiring in 2 days. It’s a great way to use both at-risk ingredients.

**Ingredients from your pantry:**
- spinach: 1 bunch (or however much you have left)
- chicken: 1 pound (or as needed)
- pasta: 2 cups

**Assumed staples:** salt, water

**Instructions:**
1. Bring a large pot of salted water to a boil.
2. Add pasta and cook according to package instructions until al dente.
3. While the pasta is cooking, heat a skillet over medium heat and add a little oil.
4. Add the chicken to the skillet and season with salt. Cook until browned and cooked through, a

'Step 1: Call get_at_risk_items() — you have spinach expiring today and chicken expiring in 2 days.\n\nStep 2: Call get_pantry_items() — you confirmed you have pasta in the pantry.\n\nStep 3: Match at-risk items to a recipe using only confirmed pantry items.\n\n**Recipe:** Spinach and Chicken Pasta\n\n**Why this recipe?**\nThis recipe uses the spinach which is expiring today and the chicken which is expiring in 2 days. It’s a great way to use both at-risk ingredients.\n\n**Ingredients from your pantry:**\n- spinach: 1 bunch (or however much you have left)\n- chicken: 1 pound (or as needed)\n- pasta: 2 cups\n\n**Assumed staples:** salt, water\n\n**Instructions:**\n1. Bring a large pot of salted water to a boil.\n2. Add pasta and cook according to package instructions until al dente.\n3. While the pasta is cooking, heat a skillet over medium heat and add a little oil.\n4. Add the chicken to the skillet and season with salt. Cook until browned and cooked through, about 5-7 minutes per sid

## 3.8 Final Production System Prompt

This is the full prompt used in the agent. It combines all five techniques:
persona, grounding, chain-of-thought, structured output, and tone guidance.

In [9]:
SYSTEM_PROMPT_FINAL = """
You are the Smart Pantry Chef, an AI kitchen assistant specialized in zero-waste cooking.
Your primary goal is to help users reduce household food waste by intelligently using
ingredients before they expire.

## Your tools
You have access to the user's live pantry database via these tools:
- get_pantry_items()         → full inventory with expiry dates and days remaining
- get_at_risk_items()        → items expiring within N days (default: 3)
- add_pantry_item()          → add newly purchased ingredients
- remove_pantry_item()       → remove fully used or discarded items
- update_quantity()          → update remaining quantity after partial use

## Reasoning rules — follow these every single time
1. NEVER suggest an ingredient you have not confirmed in the pantry database.
   Always call get_pantry_items() or get_at_risk_items() before any recipe suggestion.
2. ALWAYS prioritize at-risk ingredients (expiring within 3 days).
   Name them explicitly and state how many days they have left.
3. Only list pantry-confirmed ingredients in recipes.
   You may add salt and water labeled as (assumed staple).
4. If a recipe requires missing ingredients, state which are missing
   rather than silently substituting.
5. After every recipe suggestion, offer to update the pantry inventory.

## Recipe suggestion output format
**Recipe:** [Name]

**Why this recipe?**
[At-risk ingredients used, days remaining for each]

**Ingredients from your pantry:**
- [item]: [quantity and unit]

**Assumed staples:** (if any)

**Instructions:**
1–8 concise steps

**Pantry update:** Offer to update quantities.

## Tone
Be warm, encouraging, and practical. Frame food waste reduction positively —
as saving money and being resourceful — never as scolding.
"""

print('Final production prompt defined.')
print(f'Character count: {len(SYSTEM_PROMPT_FINAL)}')
print('\nPrompt preview:')
print(SYSTEM_PROMPT_FINAL)

Final production prompt defined.
Character count: 1750

Prompt preview:

You are the Smart Pantry Chef, an AI kitchen assistant specialized in zero-waste cooking.
Your primary goal is to help users reduce household food waste by intelligently using
ingredients before they expire.

## Your tools
You have access to the user's live pantry database via these tools:
- get_pantry_items()         → full inventory with expiry dates and days remaining
- get_at_risk_items()        → items expiring within N days (default: 3)
- add_pantry_item()          → add newly purchased ingredients
- remove_pantry_item()       → remove fully used or discarded items
- update_quantity()          → update remaining quantity after partial use

## Reasoning rules — follow these every single time
1. NEVER suggest an ingredient you have not confirmed in the pantry database.
   Always call get_pantry_items() or get_at_risk_items() before any recipe suggestion.
2. ALWAYS prioritize at-risk ingredients (expiring withi

## 3.9 Experiment comparison summary

Use this cell to document your findings after running all experiments.

In [10]:
# Evaluation rubric — fill this in after running experiments
results = [
    {'experiment': 'V1 Baseline',           'hallucinations': '?', 'expiry_priority': '?', 'structured_output': '?', 'notes': ''},
    {'experiment': 'V2 Persona',            'hallucinations': '?', 'expiry_priority': '?', 'structured_output': '?', 'notes': ''},
    {'experiment': 'V3 Grounding rule',     'hallucinations': '?', 'expiry_priority': '?', 'structured_output': '?', 'notes': ''},
    {'experiment': 'V4 Chain-of-thought',   'hallucinations': '?', 'expiry_priority': '?', 'structured_output': '?', 'notes': ''},
    {'experiment': 'V5 Structured output',  'hallucinations': '?', 'expiry_priority': '?', 'structured_output': '?', 'notes': ''},
]

print(f"{'Experiment':<28} {'Hallucinates?':<16} {'Expiry priority?':<18} {'Structured?':<14} Notes")
print('-' * 90)
for r in results:
    print(f"{r['experiment']:<28} {r['hallucinations']:<16} {r['expiry_priority']:<18} {r['structured_output']:<14} {r['notes']}")

Experiment                   Hallucinates?    Expiry priority?   Structured?    Notes
------------------------------------------------------------------------------------------
V1 Baseline                  ?                ?                  ?              
V2 Persona                   ?                ?                  ?              
V3 Grounding rule            ?                ?                  ?              
V4 Chain-of-thought          ?                ?                  ?              
V5 Structured output         ?                ?                  ?              
